|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>The block allocator<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: build the allocator and the page table<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

Build the allocator.

You need no tensors and no GPU. This is stage 06 of the ladder. It is pure
bookkeeping. Get it correct on its own, before a kernel must read through
it.

In [ ]:
### run this cell

BLOCK_SIZE = 16                # tokens per block
NUM_BLOCKS = 4096              # blocks in the pool
lengths = rng.lognormal(mean=np.log(120), sigma=0.9, size=2000).astype(int) + 1

print(f'pool holds {NUM_BLOCKS*BLOCK_SIZE:,} tokens in {NUM_BLOCKS:,} blocks of {BLOCK_SIZE}')

# Exercise 1: the free list

You need a pool of blocks, a stack of the free ones, and one reference count
per block. The reference count looks unnecessary now. Exercise 4 explains
it.

In [ ]:
class BlockAllocator:
  def __init__(self, num_blocks):
    self.free_blocks = list(range(num_blocks))      # a stack of physical ids
    self.ref_counts = [0] * num_blocks

  def allocate(self):
    if not self.free_blocks:
      raise MemoryError('out of KV blocks')
    block_id = self.free_blocks.pop()
    self.ref_counts[block_id] = 1
    return block_id

  def release(self, block_id):
    self.ref_counts[block_id] -= 1
    if self.ref_counts[block_id] == 0:
      self.free_blocks.append(block_id)

  def share(self, block_id):
    self.ref_counts[block_id] += 1
    return block_id

  @property
  def used(self):
    return len(self.ref_counts) - len(self.free_blocks)

allocator = BlockAllocator(8)
first_block, second_block = allocator.allocate(), allocator.allocate()
print('used after 2 allocations:', allocator.used)
allocator.release(first_block)
print('used after 1 release:    ', allocator.used)

# Exercise 2: the block table

Each sequence holds one table. The table maps a logical token position onto a
physical slot in the pool. It grows one block at a time, as the sequence
grows.

In [ ]:
class BlockTable:
  """One sequence's view of the pool."""
  def __init__(self, allocator, block_size):
    self.allocator = allocator
    self.block_size = block_size
    self.blocks = []      # logical block index -> physical block id
    self.num_tokens = 0       # tokens held

  def append_token(self):
    if self.num_tokens % self.block_size == 0:            # the current block is full
      self.blocks.append(self.allocator.allocate())
    self.num_tokens += 1

  def slot_index(self, pos):
    """logical token position -> flat slot in the pool"""
    return self.blocks[pos // self.block_size] * self.block_size + pos % self.block_size

  def free(self):
    for block_id in self.blocks:
      self.allocator.release(block_id)
    self.blocks, self.num_tokens = [], 0

allocator = BlockAllocator(NUM_BLOCKS)
table = BlockTable(allocator, BLOCK_SIZE)
for _ in range(40):
  table.append_token()
print(f'40 tokens -> {len(table.blocks)} blocks: {table.blocks}')
print(f'position 0  -> slot {table.slot_index(0)}')
print(f'position 17 -> slot {table.slot_index(17)}')

# Exercise 3: how many sequences fit now?

Admit requests from the workload until the allocator refuses. Then compare
your answer against a reservation of `max_len` for each request.

In [ ]:
allocator = BlockAllocator(NUM_BLOCKS)
tables = []
admitted = 0

for length in lengths:
  table = BlockTable(allocator, BLOCK_SIZE)
  try:
    for _ in range(int(length)):
      table.append_token()
  except MemoryError:
    table.free()
    break
  tables.append(table)
  admitted += 1

held = sum(len(table.blocks) for table in tables) * BLOCK_SIZE
used = sum(table.num_tokens for table in tables)
print(f'admitted {admitted} sequences before the pool ran out')
print(f'tokens held {held:,}, tokens used {used:,}  -> {100*(1-used/held):.1f}% wasted')

MAX_LEN = 2048
contiguous = (NUM_BLOCKS*BLOCK_SIZE) // MAX_LEN
print(f'\ncontiguous, reserving {MAX_LEN}: {contiguous} sequences')
print(f'paged:                     {admitted} sequences   ({admitted/contiguous:.0f}x)')

# Exercise 4: two sequences, one prompt

Parallel sampling asks the model for four replies to one prompt. All four
replies share the same K and V for the prompt.

A page table makes that free.

In [ ]:
allocator = BlockAllocator(NUM_BLOCKS)
before = allocator.used

parent = BlockTable(allocator, BLOCK_SIZE)
for _ in range(64):
  parent.append_token()
after_parent = allocator.used

# four samples from the same prompt: share every block the prompt holds
children = []
for _ in range(4):
  child = BlockTable(allocator, BLOCK_SIZE)
  child.blocks = [allocator.share(block_id) for block_id in parent.blocks]
  child.num_tokens = parent.num_tokens
  children.append(child)

print(f'prompt of 64 tokens costs      {after_parent - before} blocks')
print(f'4 samples sharing it cost      {allocator.used - after_parent} more')
print(f'4 samples copying it would be  {4*(after_parent-before)} more')

for child in children:
  child.free()
print(f'\nafter the children leave, still held: {allocator.used} blocks (the parent)')

### What you built

You built an allocator, a page table, and a reference count. That is about
sixty lines and no tensors. It gives about ten times the memory efficiency of
the code it replaces.

Remember Exercise 4. A shared prefix became **a pointer operation**. Four
samples from one prompt cost four block-table entries, not four copies of the
prompt's KV cache. The reference count also stops one sequence from freeing a
block that another sequence still reads.

Follow that idea and you reach the rest of stage 09:

- One sequence writes into a shared block. Copy the block first, for that
  sequence only. This is copy-on-write, exactly as `fork` uses it.
- Hash the contents of a block. Two unrelated **requests** with the same
  system prompt then share it. This is automatic prefix caching, which is a
  page cache.

A contiguous buffer offered none of this. Paging did not make sharing faster.
Paging made sharing possible to express.

    ./vc guide 6